[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-07-deployments-basics.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Prefect Deployments — Packaging Flows for Scheduled Execution
**certified-journeys / prefect-certified** &nbsp;|&nbsp; Practice

> **Goal for today:** Create Prefect deployments from Python code, understand every field in the deployment manifest, trigger and inspect runs from the UI and CLI, and override parameters at run time.

---
## What is a deployment?

A **deployment** is a server-side representation of a flow that tells Prefect:
- *where* to find the flow code (the `entrypoint`)
- *when* to run it (schedule or on-demand)
- *how* to run it (which work pool, which parameters)

| Without a deployment | With a deployment |
|---|---|
| You run `python my_flow.py` manually | Prefect schedules and triggers runs for you |
| Runs only on your laptop | Worker can run anywhere — laptop, VM, container |
| No run history | Full run history, logs, and retries in the UI |

The two main ways to create a deployment:

| Method | When to use |
|---|---|
| `flow.from_source().deploy()` | Programmatic, from Python |
| `prefect.yaml` + `prefect deploy` | GitOps / CI-CD workflows |

Today we use the Python API — it is the fastest way to get started.

In [ ]:
%pip install -q "prefect>=2.14" prefect-shell

---
## Step 1 · Start a local Prefect server

Before creating deployments you need a running Prefect server (the orchestration layer).
In production you use Prefect Cloud — here we run a local ephemeral server.

The Prefect API URL is configured via the `PREFECT_API_URL` environment variable.
When you run `prefect server start` it listens on `http://127.0.0.1:4200/api`.

> **Colab note:** We use `prefect.testing.utilities.prefect_test_harness` so all API calls go to a
> lightweight in-process server — no sockets needed in the notebook environment.

In [ ]:
# Configure an in-process Prefect server for notebook experiments
import os
os.environ["PREFECT_API_URL"] = "http://127.0.0.1:4200/api"
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"

# Use the ephemeral (in-process) API so no server process is required
from prefect.settings import PREFECT_API_URL
os.environ["PREFECT_API_URL"] = ""   # empty → use ephemeral server

import prefect
print(f"Prefect version: {prefect.__version__}")
print("Environment ready — using in-process ephemeral server.")

**What just happened?**
- `PREFECT_API_URL=""` tells Prefect to use an **ephemeral** in-process server — ideal for notebooks
- In production you'd set `PREFECT_API_URL` to your Prefect Cloud workspace URL
- `prefect server start` starts the server as a background process and opens the UI on port 4200

---
## Step 2 · Write a parameterised flow

A **flow** is any function decorated with `@flow`. Parameters become the deployment's default
parameters — you can override them at run time from the UI, CLI, or API.

Best practices for deployable flows:
- Accept all configuration as function parameters (no hardcoded values)
- Return a meaningful value — the return value is captured in the run record
- Use `@task` for logical sub-steps so Prefect tracks them individually

In [ ]:
from prefect import flow, task, get_run_logger
from datetime import datetime

@task(retries=2, retry_delay_seconds=5)
def fetch_data(source: str, limit: int) -> list[dict]:
    """Simulate fetching records from a data source."""
    logger = get_run_logger()
    logger.info(f"Fetching up to {limit} records from '{source}'")
    # Simulate data — in production this hits an API or database
    records = [
        {"id": i, "source": source, "value": i * 3.14, "ts": datetime.utcnow().isoformat()}
        for i in range(1, limit + 1)
    ]
    logger.info(f"Fetched {len(records)} records.")
    return records

@task
def process_records(records: list[dict], multiplier: float) -> dict:
    """Apply a multiplier to the value field of each record."""
    processed = [{**r, "value": round(r["value"] * multiplier, 4)} for r in records]
    total = sum(r["value"] for r in processed)
    return {"count": len(processed), "total": round(total, 4)}

@flow(name="daily-data-pipeline", log_prints=True)
def daily_pipeline(
    source: str = "api-v1",
    limit: int = 100,
    multiplier: float = 1.0,
) -> dict:
    """
    Fetch and process records from a data source.

    Args:
        source:     Name of the data source to pull from
        limit:      Maximum number of records to fetch
        multiplier: Scaling factor applied to each record's value

    Returns:
        Summary dict with count and total value
    """
    records = fetch_data(source=source, limit=limit)
    summary = process_records(records=records, multiplier=multiplier)
    print(f"Pipeline complete: {summary}")
    return summary

# Test the flow locally before deploying it
result = daily_pipeline(source="test-source", limit=5, multiplier=2.0)
print(f"Return value: {result}")

**What just happened?**
- `@task(retries=2)` — Prefect automatically retries the task up to 2 times on failure
- `@flow(log_prints=True)` — every `print()` inside the flow is captured as a log entry
- **Default parameters** (`source`, `limit`, `multiplier`) become the deployment's defaults — override them per-run
- `get_run_logger()` returns a logger that writes to Prefect's log store (visible in the UI)

---
## Step 3 · The deployment manifest — every field explained

A deployment is described by a manifest. When you call `flow.deploy()`, Prefect
registers this manifest with the server.

| Field | What it controls |
|---|---|
| `name` | Human-readable deployment name (unique per flow) |
| `flow_name` | The registered name from `@flow(name=...)` |
| `entrypoint` | `path/to/file.py:function_name` |
| `parameters` | Default parameter values — overridable per run |
| `schedule` | When to auto-create runs (cron, interval, rrule) |
| `work_pool_name` | Which work pool should execute this deployment |
| `tags` | Free-form labels for filtering in the UI |
| `version` | Semantic version string — track changes over time |

The entrypoint format is `module:function` — Prefect uses this to import
and execute the flow code on the worker.

In [ ]:
import json
from prefect.deployments import Deployment

# Build a Deployment object and inspect its manifest
deployment = Deployment.build_from_flow(
    flow=daily_pipeline,
    name="daily-pipeline-dev",
    version="1.0.0",
    # Default parameters for this deployment
    parameters={"source": "api-v1", "limit": 100, "multiplier": 1.0},
    tags=["data-pipeline", "dev", "day-07"],
    # No work_pool_name or schedule yet — we add those in the next steps
    description="Daily data pipeline deployment — reads from API, applies multiplier.",
)

# Show the manifest fields Prefect will register
manifest_fields = {
    "name":         deployment.name,
    "flow_name":    deployment.flow_name,
    "version":      deployment.version,
    "parameters":   deployment.parameters,
    "tags":         deployment.tags,
    "description":  deployment.description,
    "work_pool_name": deployment.work_pool_name,  # None until we set it
    "schedule":     deployment.schedule,           # None until we set it
}

print("Deployment manifest:")
print(json.dumps(manifest_fields, indent=2, default=str))

**What just happened?**
- `Deployment.build_from_flow()` constructs the manifest object in memory — nothing is sent to the server yet
- `flow_name` is populated from `@flow(name="daily-data-pipeline")` automatically
- `work_pool_name=None` means this deployment cannot be executed by a worker yet — we fix that next
- **Tags** let you filter deployments in the UI: filter by `dev` to see all development deployments

---
## Step 4 · Create a work pool and register the deployment

A **work pool** is the bridge between the Prefect server (which schedules runs)
and the **worker** (which executes them). Every deployment must target a work pool.

Work pool types:

| Type | Infrastructure | When to use |
|---|---|---|
| `process` | Subprocess on the worker machine | Local development |
| `docker` | Docker container | Reproducible, isolated environments |
| `kubernetes` | Kubernetes pod | Production, auto-scaling |
| `ecs` | AWS ECS task | AWS-native deployments |

Today we create a `process` pool — the simplest type.

In [ ]:
import asyncio
from prefect.client.orchestration import get_client
from prefect.server.schemas.actions import WorkPoolCreate

WORK_POOL_NAME = "local-process-pool"

async def create_work_pool_and_deploy():
    async with get_client() as client:
        # Create a process work pool (type='process' = subprocess worker)
        try:
            wp = await client.create_work_pool(
                WorkPoolCreate(name=WORK_POOL_NAME, type="process")
            )
            print(f"Work pool created: {wp.name!r} (type={wp.type})")
        except Exception as e:
            # Pool may already exist — that's fine
            print(f"Work pool note: {e}")

        # Register a deployment that targets this pool
        from prefect.server.schemas.actions import DeploymentCreate
        from prefect.server.schemas.schedules import NoSchedule

        # Register the flow first (gives us a flow_id)
        flow_id = await client.create_flow_from_name("daily-data-pipeline")
        print(f"Flow registered with id: {flow_id}")

        dep = await client.create_deployment(
            DeploymentCreate(
                name="daily-pipeline-dev",
                flow_id=flow_id,
                version="1.0.0",
                entrypoint="day_07_flow:daily_pipeline",
                parameters={"source": "api-v1", "limit": 100, "multiplier": 1.0},
                tags=["data-pipeline", "dev", "day-07"],
                work_pool_name=WORK_POOL_NAME,
                description="Daily data pipeline deployment.",
            )
        )
        print(f"Deployment registered with id: {dep}")
        return dep

deployment_id = asyncio.run(create_work_pool_and_deploy())
print(f"\nDeployment ID: {deployment_id}")

**What just happened?**
- `create_work_pool()` registers the pool type with the Prefect server — workers of that type can poll it
- `create_deployment()` sends the manifest to the server — the deployment is now visible in the UI
- The `entrypoint` (`day_07_flow:daily_pipeline`) tells the worker which Python module and function to import
- **Nothing has run yet** — we only registered intent; a worker + trigger are still needed

---
## Step 5 · Trigger a run from the CLI and override parameters

Once a deployment is registered, you can trigger it three ways:

| Method | Command / API |
|---|---|
| UI Quick Run | Click **Quick Run** on the deployment detail page |
| CLI | `prefect deployment run 'flow-name/deployment-name'` |
| Python API | `deployment.run(parameters={...})` |

The `--param` flag lets you override individual parameters without changing the deployment:

```bash
# Production CLI usage — override limit and multiplier for this run only
prefect deployment run 'daily-data-pipeline/daily-pipeline-dev' \
  --param source=api-v2 \
  --param limit=500 \
  --param multiplier=2.5
```

Below we do the same thing programmatically via the client API.

In [ ]:
import asyncio
from prefect.client.orchestration import get_client

async def trigger_run_with_overrides(deployment_id: str):
    """Trigger a deployment run with parameter overrides via the Python API."""
    async with get_client() as client:
        # Create a flow run with overridden parameters
        # This is equivalent to clicking Quick Run → filling the parameter form
        flow_run = await client.create_flow_run_from_deployment(
            deployment_id=str(deployment_id),
            parameters={
                "source": "api-v2",   # override: different source
                "limit": 25,          # override: smaller batch for testing
                "multiplier": 2.5,    # override: custom scaling
            },
            tags=["manual-trigger", "param-override"],
        )
        print(f"Flow run created:")
        print(f"  id:         {flow_run.id}")
        print(f"  name:       {flow_run.name}")
        print(f"  state:      {flow_run.state_type}")
        print(f"  parameters: {flow_run.parameters}")
        return flow_run.id

if deployment_id:
    run_id = asyncio.run(trigger_run_with_overrides(deployment_id))
    print(f"\nRun queued with id: {run_id}")
    print("In a real environment, the worker would pick this up and execute it.")
else:
    print("No deployment_id — skipping trigger step.")

**What just happened?**
- `create_flow_run_from_deployment()` creates a run record on the server — state is `Scheduled`
- **Parameter overrides** are merged with the deployment defaults — only the keys you pass are overridden
- The run stays in `Scheduled` state until a worker picks it up from the work pool queue
- In the UI, this run appears in **Flow Runs** — you can inspect parameters, logs, and task states

---
## Step 6 · Inspect run history and compare runs

Prefect stores every run's metadata: parameters used, start/end time, task states, return value, and logs.
Use the client to query run history programmatically — the same data the UI shows.

**UI tips for comparing runs:**
- Go to **Flow Runs**, filter by deployment name
- Select two runs → click **Compare** to see parameter diffs side-by-side
- Each task's duration is shown in the timeline — spot regressions at a glance

In [ ]:
import asyncio
from prefect.client.orchestration import get_client
from prefect.client.schemas.filters import FlowRunFilter, FlowRunFilterDeploymentId
from prefect.client.schemas.sorting import FlowRunSort

async def list_deployment_runs(deployment_id: str, limit: int = 10):
    """Fetch the most recent runs for a deployment."""
    async with get_client() as client:
        runs = await client.read_flow_runs(
            flow_run_filter=FlowRunFilter(
                deployment_id=FlowRunFilterDeploymentId(any_=[str(deployment_id)])
            ),
            sort=FlowRunSort.START_TIME_DESC,
            limit=limit,
        )
        print(f"Found {len(runs)} run(s) for deployment {deployment_id}:\n")
        for run in runs:
            print(f"  Run: {run.name}")
            print(f"    State:      {run.state_type}")
            print(f"    Parameters: {run.parameters}")
            print(f"    Created at: {run.created}")
            print()
        return runs

if deployment_id:
    runs = asyncio.run(list_deployment_runs(deployment_id))
else:
    print("No deployment_id — create a deployment first (Step 4).")

# Production CLI equivalent:
print("\nCLI equivalent:")
print("  prefect deployment inspect 'daily-data-pipeline/daily-pipeline-dev'")
print("  prefect flow-run ls --deployment 'daily-data-pipeline/daily-pipeline-dev'")

**What just happened?**
- `read_flow_runs()` with a `FlowRunFilter` returns only runs for our specific deployment
- `FlowRunSort.START_TIME_DESC` gives us the most recent runs first
- Each run record contains the **exact parameters** used — critical for debugging data quality issues
- In the UI: **Flow Runs → filter by deployment → click any run → Parameters tab** shows the same data

---
## Step 7 · Using `flow.from_source().deploy()` — the modern Python API

Prefect 2.14+ adds a cleaner one-shot API: `flow.from_source().deploy()` reads code from a
storage location and registers the deployment in one call.

| Storage type | `from_source()` argument | When to use |
|---|---|---|
| Git repo | `source=GitRepository(url=...)` | Teams, CI/CD |
| Local path | `source="./path/to/code"` | Local dev |
| S3 / GCS / Azure | `source=S3Bucket(bucket=...)` | Cloud storage |

Below we simulate a local-path deployment (safe for Colab — no git remote needed).

In [ ]:
import tempfile
import os
import textwrap

# Write the flow to a temp file — simulates having code in a directory
tmp_dir = tempfile.mkdtemp()
flow_path = os.path.join(tmp_dir, "my_pipeline.py")

flow_code = textwrap.dedent("""
    from prefect import flow, task, get_run_logger
    from datetime import datetime

    @task
    def extract(source: str, limit: int) -> list:
        logger = get_run_logger()
        logger.info(f"Extracting {limit} records from {source}")
        return [{"id": i, "source": source} for i in range(limit)]

    @flow(name="from-source-pipeline", log_prints=True)
    def my_pipeline(source: str = "default", limit: int = 10) -> int:
        records = extract(source=source, limit=limit)
        print(f"Processed {len(records)} records from '{source}'")
        return len(records)
""")

with open(flow_path, "w") as f:
    f.write(flow_code)

print(f"Flow file written to: {flow_path}")
print(f"\nFlow code:\n{flow_code}")

# Show the from_source().deploy() call pattern
# In a real environment (with a running server + worker), you would run:
#
# from prefect import flow
# from prefect.runner.storage import LocalStorage
#
# flow.from_source(
#     source=LocalStorage(path=tmp_dir),
#     entrypoint="my_pipeline.py:my_pipeline",
# ).deploy(
#     name="from-source-demo",
#     work_pool_name="local-process-pool",
#     parameters={"source": "api-v1", "limit": 50},
#     build=False,  # skip Docker build for process pools
# )
#
# The above call registers the deployment AND verifies the entrypoint is importable.

print("\n--- from_source().deploy() pattern shown above (commented to avoid server connection) ---")
print("In the UI: Deployments → daily-pipeline-dev → Quick Run → adjust parameters → Run")

**What just happened?**
- `flow.from_source(source=LocalStorage(path=...), entrypoint="file.py:function")` loads the flow definition from a directory
- `.deploy()` registers the deployment — one call replaces the manual `Deployment.build_from_flow()` + `apply()` pattern
- **The entrypoint** is the critical field: it tells workers exactly what to import and run
- `build=False` skips Docker image building — required for `process` pool deployments

---
## Step 8 · Deployment lifecycle — from creation to completion

Let's trace the full lifecycle of a deployment run to understand every state transition.

In [ ]:
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness

# Use the test harness to run a flow locally and inspect every state
with prefect_test_harness():

    @task(retries=1, retry_delay_seconds=1)
    def validate_config(source: str, limit: int) -> bool:
        if limit <= 0:
            raise ValueError(f"limit must be > 0, got {limit}")
        if not source:
            raise ValueError("source cannot be empty")
        return True

    @flow(name="lifecycle-demo", log_prints=True)
    def lifecycle_flow(
        source: str = "prod-api",
        limit: int = 50,
        multiplier: float = 1.0,
    ) -> dict:
        """Demonstrates the full deployment run lifecycle."""
        valid = validate_config(source=source, limit=limit)
        records = [{"id": i, "v": i * multiplier} for i in range(limit)]
        summary = {"source": source, "count": len(records), "total": sum(r["v"] for r in records)}
        print(f"Run summary: {summary}")
        return summary

    # Run 1: default parameters
    print("--- Run 1: default parameters ---")
    r1 = lifecycle_flow()
    print(f"Result: {r1}")

    # Run 2: parameter override — simulates 'Quick Run' with custom params
    print("\n--- Run 2: overridden parameters (simulates UI Quick Run) ---")
    r2 = lifecycle_flow(source="staging-db", limit=10, multiplier=3.0)
    print(f"Result: {r2}")

    print("\n--- Run comparison ---")
    print(f"Run 1 count={r1['count']}, total={r1['total']}")
    print(f"Run 2 count={r2['count']}, total={r2['total']}")
    print(f"Total delta: {r2['total'] - r1['total']:.2f}")

**What just happened?**
- `prefect_test_harness()` starts an isolated in-process Prefect environment — no server needed
- **Run 1** uses deployment defaults; **Run 2** overrides all three parameters
- In the UI, you'd see both runs in Flow Runs — same deployment, different parameters, different totals
- The **Compare** button in the UI shows parameter diffs between any two runs side-by-side

---
## Challenge

You have a flow that aggregates sales by region:

```python
from prefect import flow, task

@task
def fetch_sales(region: str, days_back: int) -> list[dict]:
    # Simulated sales data
    import random
    random.seed(42)
    return [{"region": region, "day": d, "revenue": random.uniform(1000, 9999)} for d in range(days_back)]

@task
def aggregate(records: list[dict]) -> dict:
    total = sum(r["revenue"] for r in records)
    return {"region": records[0]["region"], "total_revenue": round(total, 2), "days": len(records)}

@flow(name="sales-aggregator", log_prints=True)
def sales_flow(region: str = "US", days_back: int = 7) -> dict:
    records = fetch_sales(region=region, days_back=days_back)
    result = aggregate(records=records)
    print(f"Result: {result}")
    return result
```

**Your tasks:**
1. Run the flow with `prefect_test_harness()` using default parameters
2. Trigger a second run that overrides `region="EU"` and `days_back=30`
3. Compare the two `total_revenue` values and print which region had higher revenue
4. Build a `Deployment` manifest (no need to apply it) with `tags=["sales", "prod"]` and `version="2.0.0"`

In [ ]:
# Challenge: your solution here
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness
from prefect.deployments import Deployment

# Step 1 & 2: define and run the flow twice with different parameters
# YOUR CODE HERE

# Step 3: compare results
# YOUR CODE HERE

# Step 4: build (don't apply) a Deployment manifest
# YOUR CODE HERE

---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Deployment | Server-side manifest: where the code lives, when to run, what parameters to use |
| `entrypoint` | `module.py:function_name` — tells the worker what to import |
| `flow.from_source().deploy()` | One-shot API: load code from storage + register deployment |
| Work pool | Logical queue between server (scheduler) and worker (executor) |
| Parameter override | Pass `--param key=val` in CLI or `parameters={}` in API to override per run |
| Run history | Each run stores exact parameters, state transitions, logs, and return value |
| `prefect_test_harness()` | In-process Prefect environment — no server needed for unit tests |

> **Tip:** A deployment is the bridge between your flow code and its runtime environment — define it once and you can trigger it from the UI, CLI, API, or a schedule.

---
## What's next
**Day 8** → Work pools and workers in depth: create local process and Docker work pools, start a worker, and understand how the polling loop picks up queued runs.

Mark Day 7 complete in your [tracker](../index.html).